## Q1. Null and Alternative Hypothesis

**Scenario:** I believe people spend more time on Instagram than on YouTube daily.

Let μ_I = average daily time on Instagram, μ_Y = average daily time on YouTube.

- **H0 (null):** μ_I ≤ μ_Y — people do *not* spend more time on Instagram than on YouTube.
- **H1 (alternative):** μ_I > μ_Y — people spend *more* time on Instagram than on YouTube.

This is a **one-tailed (right-tailed)** test because the belief has a direction.

## Q2. Two-sample t-test (Zomato home page)

| Group | Page | Avg order value | Std dev | Users |
|---|---|---|---|---|
| A | New | ₹350 | ₹50 | 30 |
| B | Old | ₹320 | ₹50 | 30 |

- **H0:** μ_A = μ_B (new page makes no difference to average order value)
- **H1:** μ_A ≠ μ_B (new page changes average order value)
- **Significance level:** α = 0.05

In [1]:
from scipy import stats

t, p = stats.ttest_ind_from_stats(mean1=350, std1=50, nobs1=30,
                                  mean2=320, std2=50, nobs2=30)

print(f"t-statistic = {t:.3f}")
print(f"p-value     = {p:.4f}")
print("Decision:", "Reject H0" if p < 0.05 else "Fail to reject H0")

t-statistic = 2.324
p-value     = 0.0237
Decision: Reject H0


**Conclusion:** The p-value (≈ 0.024) is less than 0.05, so we **reject the null hypothesis**.
The new home page leads to a statistically significant increase in average order value (₹350 vs ₹320).

*(Note: we never "accept" H0 — we either reject it or fail to reject it.)*

## Q3. Designing an A/B Test — Paytm Redesigned Payment Button

**Feature:** A redesigned "Pay Now" button (new colour, larger size, clearer label) on the payment screen.

**Groups**
- **Control (A):** Users who see the existing "Pay Now" button.
- **Variant (B):** Users who see the redesigned button.
- Users are **randomly assigned 50/50** (e.g. by hashing user ID) so both groups are similar. Each user always sees the same version.

**Primary metric:** Payment conversion rate = (users who complete a payment) / (users who reach the payment screen).

**Secondary / guardrail metrics:** payment failure rate, time taken to complete payment, app crash rate. These make sure the new button isn't hurting anything else.

**Hypotheses**
- **H0:** p_B = p_A — the conversion rate is the same for both buttons.
- **H1:** p_B ≠ p_A — the conversion rate differs between the buttons (use p_B > p_A if only an improvement matters).

**Test setup**
1. Fix α = 0.05 and decide the minimum improvement worth detecting (e.g. +1 percentage point) *before* the test.
2. Calculate the required sample size per group.
3. Run for at least 1–2 full weeks to cover weekday/weekend behaviour; don't stop early when the result "looks good".
4. Compare conversion rates with a two-proportion z-test. If p < 0.05, reject H0 and roll out the new button.

## Q4. Flipkart 'Add to Wishlist' Button

**Setup (assumption):** 1000 users saw each version. 520 clicked the new button (52%) vs 480 clicked the old one (48%).

**Intuition (no detailed calculation):**
The difference is only 4 percentage points (40 clicks out of 1000). With samples of this size, random variation alone can easily move click counts by ±20 or so, so a 40-click gap is *borderline*. It is **probably not strong enough** to call significant at the 5% level — it could well be due to chance.

**How the p-value helps:**
The p-value is the probability of seeing a difference at least this large *if the two buttons were actually equally good* (H0 true).
- Small p-value (< 0.05): such a gap would be rare by luck alone, so we reject H0 and call the result significant.
- Large p-value (≥ 0.05): the gap is plausible by chance, so we do not have enough evidence to say the new button is better.

Quick check below (optional):

In [2]:
import numpy as np
from scipy import stats

n = 1000
p_new, p_old = 520 / n, 480 / n
p_pool = (520 + 480) / (2 * n)

z = (p_new - p_old) / np.sqrt(p_pool * (1 - p_pool) * (2 / n))
p = 2 * (1 - stats.norm.cdf(z))

print(f"z = {z:.3f}, p-value = {p:.4f}")
print("Significant" if p < 0.05 else "Not significant at 0.05")

z = 1.789, p-value = 0.0736
Not significant at 0.05


**Result:** p ≈ 0.07, which is above 0.05, so the difference is **not statistically significant** — it could be just chance. To be more confident, we would collect more data (a larger sample) before declaring the new button the winner.

*(If instead 1000 total clicks were split 520 vs 480 between the two buttons, a binomial test gives an even larger p-value, p ≈ 0.2 — the same conclusion.)*